# Combine Provider Datasets → Master Final Dataset

Merges all 6 provider-level final datasets into a single `master_final_dataset.csv`.

| Provider | File | Rows |
|---|---|---|
| Groq | `groq_final_dataset.csv` | 180 |
| Cerebras | `cerebras_final_dataset.csv` | 120 |
| SambaNova | `sambanova_final_dataset.csv` | 180 |
| Mistral | `mistral_final_dataset.csv` | 180 |
| DeepSeek | `deepseek_final_dataset.csv` | 120 |
| Cohere | `cohere_final_dataset.csv` | 180 |
| **Total** | | **960** |

## Cell 1 — Imports

In [1]:
import pandas as pd
import os

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 50)


## Cell 2 — Load & Combine All Provider Datasets

In [2]:
# Place all provider final CSV files in the same folder as this notebook
PROVIDER_FILES = [
    'groq_final_dataset.csv',
    'cerebras_final_dataset.csv',
    'sambanova_final_dataset.csv',
    'mistral_final_dataset.csv',
    'deepseek_final_dataset.csv',
    'cohere_final_dataset.csv',
]

dfs = []
for fname in PROVIDER_FILES:
    if not os.path.exists(fname):
        print(f'  WARNING: {fname} not found — skipping')
        continue
    df = pd.read_csv(fname)
    dfs.append(df)
    print(f'  Loaded {fname:<40} {df.shape[0]} rows')

master = pd.concat(dfs, ignore_index=True)
print(f'\nCombined shape : {master.shape}')
print(f'Expected       : (960, 25)')


  Loaded groq_final_dataset.csv                   180 rows
  Loaded cerebras_final_dataset.csv               120 rows
  Loaded sambanova_final_dataset.csv              180 rows
  Loaded mistral_final_dataset.csv                180 rows
  Loaded deepseek_final_dataset.csv               120 rows
  Loaded cohere_final_dataset.csv                 180 rows

Combined shape : (960, 25)
Expected       : (960, 25)


## Cell 3 — Validation Checks

In [3]:
print('=== Row Count ===')
print(f'Total rows  : {len(master)}  (expected 960)')
print(f'Total cols  : {len(master.columns)}  (expected 25)')

print()
print('=== Rows per Provider ===')
print(master['Provider'].value_counts().to_string())

print()
print('=== Rows per Model (should be 60 each) ===')
model_counts = master['Model'].value_counts()
print(model_counts.to_string())
if (model_counts == 60).all():
    print('\n✓ All 16 models have exactly 60 rows')
else:
    print('\n⚠ Some models do not have 60 rows — check above')

print()
print('=== Y/N Distribution ===')
print(master['Y/N'].value_counts().to_string())

print()
print('=== Field Coverage ===')
PERSONA_FIELDS = [
    'Name', 'Age', 'Gender', 'Personality Traits', 'Domain of Work',
    'Years of Exp.', 'Location', 'Education Level',
    'Devices and Technologies Use', 'Reason(s)', 'Y/N'
]
for col in PERSONA_FIELDS:
    n   = master[col].notna().sum()
    pct = n / len(master) * 100
    bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
    print(f'  {col:<42} {bar} {n:>3}/960 ({pct:5.1f}%)')

print()
print('=== Duplicate Rows Check ===')
dupes = master.duplicated().sum()
print(f'Duplicate rows: {dupes}  (expected 0)')


=== Row Count ===
Total rows  : 960  (expected 960)
Total cols  : 25  (expected 25)

=== Rows per Provider ===
Provider
groq         180
sambanova    180
mistral      180
cohere       180
cerebras     120
deepseek     120

=== Rows per Model (should be 60 each) ===
Model
llama-3.1-8b-instant                         60
llama-3.3-70b-versatile                      60
meta-llama/llama-4-scout-17b-16e-instruct    60
llama3.1-8b                                  60
qwen-3-235b-a22b-instruct-2507               60
DeepSeek-V3.1                                60
Meta-Llama-3.3-70B-Instruct                  60
gpt-oss-120b                                 60
mistral-small-latest                         60
open-mistral-nemo                            60
open-mixtral-8x22b                           60
deepseek-chat                                60
deepseek-reasoner                            60
command-r-08-2024                            60
command-r-plus-08-2024                       60
command-

## Cell 4 — Add Global Row ID

In [4]:
# Add a unique row identifier across the full dataset
master.insert(0, 'row_id', range(1, len(master) + 1))

print(f'row_id added: 1 to {len(master)}')
print(master[['row_id', 'Provider', 'Model', 'Persona ID', 'Name', 'Y/N']].head(6).to_string(index=False))


row_id added: 1 to 960
 row_id Provider                Model Persona ID            Name Y/N
      1     groq llama-3.1-8b-instant      P1_A1 Maria Hernandez   Y
      2     groq llama-3.1-8b-instant      P1_A2  Kaito Nakamura   N
      3     groq llama-3.1-8b-instant      P1_A3     Leila Patel   N
      4     groq llama-3.1-8b-instant      P1_A1 Maria Hernandez   N
      5     groq llama-3.1-8b-instant      P1_A2  Kaito Nakamura   N
      6     groq llama-3.1-8b-instant      P1_A3     Leila Patel   Y


## Cell 5 — Dataset Overview

In [5]:
print('=== Y/N by Model ===')
yn_pivot = master.groupby('Model')['Y/N'].value_counts().unstack(fill_value=0)
# Reorder columns if they exist
col_order = [c for c in ['Y', 'N', 'N/A'] if c in yn_pivot.columns]
print(yn_pivot[col_order].to_string())

print()
print('=== Age Distribution ===')
print(master['Age'].describe().round(1).to_string())

print()
print('=== Gender Distribution ===')
print(master['Gender'].value_counts().to_string())

print()
print('=== Top 10 Locations ===')
print(master['Location'].value_counts().head(10).to_string())

print()
print('=== All Columns ===')
for i, col in enumerate(master.columns, 1):
    null_count = master[col].isna().sum()
    print(f'  {i:>2}. {col:<42} nulls: {null_count}')


=== Y/N by Model ===
Y/N                                         Y   N
Model                                            
DeepSeek-V3.1                              20  40
Meta-Llama-3.3-70B-Instruct                20  40
command-r-08-2024                          20  40
command-r-plus-08-2024                     11  22
command-r7b-12-2024                        16  32
deepseek-chat                              20  40
deepseek-reasoner                          20  40
gpt-oss-120b                                7  14
llama-3.1-8b-instant                       20  40
llama-3.3-70b-versatile                    20  40
llama3.1-8b                                20  40
meta-llama/llama-4-scout-17b-16e-instruct  20  40
mistral-small-latest                       20  40
open-mistral-nemo                          20  40
open-mixtral-8x22b                         20  40
qwen-3-235b-a22b-instruct-2507             17  34

=== Age Distribution ===
count    960.0
mean      32.5
std        9.9
min     

## Cell 6 — Save Master Dataset

In [6]:
output_path = 'master_final_dataset.csv'
master.to_csv(output_path, index=False)

print(f'Saved  : {output_path}')
print(f'Rows   : {len(master)}')
print(f'Columns: {len(master.columns)}')


Saved  : master_final_dataset.csv
Rows   : 960
Columns: 26
